# Statistical Annotations

Use this page for ggpubr-style statistical annotations: compute tests inside a
plot, compute a table first, or place labels manually.


In [ ]:
import numpy as np
import pandas as pd
from plotnine_extra import *
from plotnine_extra.data import ToothGrowth, flights, iris, penguins

tooth = ToothGrowth.assign(dose=ToothGrowth["dose"].astype(str))
penguin_data = penguins.dropna(subset=[
    "bill_length_mm",
    "bill_depth_mm",
    "body_mass_g",
    "species",
]).copy()


In [ ]:
tooth = ToothGrowth.assign(dose=ToothGrowth["dose"].astype(str))
tooth.head()


## Calculate a table before plotting

`compare_means()` is useful when the statistical table should be inspected or
saved before annotation.


In [ ]:
compare_means("len ~ dose", tooth, method="t.test")


## Add a global test label

`stat_compare_means()` places one global test label by default.


In [ ]:
(
    ggboxplot(tooth, "dose", "len", fill="dose", add="jitter")
    + stat_compare_means(method="anova", label="p.format")
    + scale_fill_tableau(k=3)
    + theme_pubr()
    + labs(x="Dose", y="Tooth length")
)


## Add pairwise brackets

`stat_pwc()` computes pairwise comparisons and draws brackets. Use
`label="p.signif"` for compact significance symbols.


In [ ]:
(
    ggboxplot(tooth, "dose", "len", fill="dose", add="quasirandom")
    + stat_pwc(method="t.test", label="p.signif", p_adjust_method="holm")
    + scale_fill_colorblind(k=3)
    + theme_classic2()
    + labs(x="Dose", y="Tooth length")
)


## Paired data need an explicit subject id

For paired tests, pass `wid`. The stat requires complete, non-duplicated
subject/group pairs.


In [ ]:
paired = pd.DataFrame({
    "subject": ["s1", "s1", "s2", "s2", "s3", "s3", "s4", "s4"],
    "time": ["before", "after"] * 4,
    "score": [4.2, 5.1, 3.8, 4.7, 5.0, 6.1, 4.5, 5.0],
})

(
    ggpaired(paired, x="time", y="score", id="subject")
    + stat_pwc(method="t.test", paired=True, wid="subject", comparisons=[("before", "after")])
    + theme_clean()
    + labs(x=None, y="Score")
)


## Place pre-computed labels manually

`stat_pvalue_manual()` accepts string group labels. Use `x_levels` when the plot
order must be explicit.


In [ ]:
pvals = pd.DataFrame({
    "group1": ["0.5", "1.0"],
    "group2": ["1.0", "2.0"],
    "p": [0.041, 0.003],
    "y.position": [35, 39],
})

(
    ggboxplot(tooth, "dose", "len", fill="dose")
    + stat_pvalue_manual(pvals, label="p.signif", x_levels=["0.5", "1.0", "2.0"])
    + scale_fill_few(k=3)
    + theme_pubr()
)


## Formatting helpers

The p-value helpers format table output and custom annotation labels.


In [ ]:
[
    format_p_value(0.00042, style="p.format.signif"),
    create_p_label(0.031),
    list_p_format_styles(),
]
